In [7]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import numpy as np
import os

print("Loading data...")
csv_path = 'all_temperature_cleaned.csv'
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
df['datetime'] = pd.to_datetime(df['timestamp'] + ' ' + df['time'], format='%Y-%m-%d %H:%M')
print("Data loaded successfully.\n")

regions = ['Rakhiyal', 'Bopal', 'Ambawadi', 'Chandkheda', 'Vastral']

# Split train (2019-2023) and test (2024)
train_df = df[(df['datetime'].dt.year >= 2019) & (df['datetime'].dt.year <= 2023)].copy()
test_df = df[df['datetime'].dt.year == 2024].copy()

output_folder_2025 = 'sarima_predictions_2025'
os.makedirs(output_folder_2025, exist_ok=True)

current_dir = os.getcwd()
metrics_list = []

# Fixed SARIMA parameters based on your best RMSE results
order = (0, 0, 0)
seasonal_order = (0, 1, 1, 24)

for region in regions:
    print(f"Processing region: {region}")

    train_series = train_df.set_index('datetime')[region].interpolate(method='time').astype('float32')
    test_series = test_df.set_index('datetime')[region].interpolate(method='time').astype('float32')

    print(f"Fitting SARIMA model with order={order} and seasonal_order={seasonal_order} for {region} using low_memory=True...")
    model = SARIMAX(
        train_series,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    # Use low_memory=True to reduce memory footprint during fitting
    fit = model.fit(disp=False, low_memory=True)
    print("Model fitted successfully.")

    print(f"Forecasting 2024 test data for {region}...")
    pred_2024 = fit.forecast(steps=len(test_series))
    mse = mean_squared_error(test_series, pred_2024)
    rmse_2024 = np.sqrt(mse)
    print(f"RMSE on 2024 test data for {region}: {rmse_2024:.4f}")

    pred_2024_df = pd.DataFrame({
        'date': test_series.index.date,
        'hour': test_series.index.hour,
        'predicted_temperature': pred_2024.values,
        'actual_temperature': test_series.values
    })
    filename_2024 = f"sarima_{region.lower()}_2024.csv"
    pred_2024_df.to_csv(os.path.join(current_dir, filename_2024), index=False)
    print(f"Saved 2024 predictions for {region} as {filename_2024}")

    print(f"Forecasting full year 2025 hourly data for {region}...")
    date_range_2025 = pd.date_range(start='2025-01-01 00:00', end='2025-12-31 23:00', freq='H')
    pred_2025 = fit.forecast(steps=len(date_range_2025))

    pred_2025_df = pd.DataFrame({
        'date': date_range_2025.date,
        'hour': date_range_2025.hour,
        'predicted_temperature': pred_2025.values
    })
    filename_2025 = f"sarima_{region.lower()}_2025.csv"
    pred_2025_df.to_csv(os.path.join(output_folder_2025, filename_2025), index=False)
    print(f"Saved 2025 predictions for {region} as {filename_2025}\n")

    metrics_list.append({
        'region': region,
        'order': order,
        'seasonal_order': seasonal_order,
        'rmse_2024': rmse_2024
    })

print("Saving all regions metrics...")
metrics_df = pd.DataFrame(metrics_list)
metrics_filename = 'sarima_model_metrics_2024.csv'
metrics_df.to_csv(os.path.join(current_dir, metrics_filename), index=False)
print(f"All regions tuning metrics saved as {metrics_filename}")
print("Process completed successfully.")


Loading data...
Data loaded successfully.

Processing region: Rakhiyal
Fitting SARIMA model with order=(0, 0, 0) and seasonal_order=(0, 1, 1, 24) for Rakhiyal using low_memory=True...


C:\Users\PARIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
C:\Users\PARIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


Model fitted successfully.
Forecasting 2024 test data for Rakhiyal...
RMSE on 2024 test data for Rakhiyal: 7.6830
Saved 2024 predictions for Rakhiyal as sarima_rakhiyal_2024.csv
Forecasting full year 2025 hourly data for Rakhiyal...


C:\Users\PARIN\AppData\Local\Temp\ipykernel_44168\509016717.py:65: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  date_range_2025 = pd.date_range(start='2025-01-01 00:00', end='2025-12-31 23:00', freq='H')


Saved 2025 predictions for Rakhiyal as sarima_rakhiyal_2025.csv

Processing region: Bopal
Fitting SARIMA model with order=(0, 0, 0) and seasonal_order=(0, 1, 1, 24) for Bopal using low_memory=True...


...

Saved 2025 predictions for Chandkheda as sarima_chandkheda_2025.csv

Processing region: Vastral
Fitting SARIMA model with order=(0, 0, 0) and seasonal_order=(0, 1, 1, 24) for Vastral using low_memory=True...


C:\Users\PARIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
C:\Users\PARIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


Model fitted successfully.
Forecasting 2024 test data for Vastral...
RMSE on 2024 test data for Vastral: 7.6708
Saved 2024 predictions for Vastral as sarima_vastral_2024.csv
Forecasting full year 2025 hourly data for Vastral...


C:\Users\PARIN\AppData\Local\Temp\ipykernel_44168\509016717.py:65: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  date_range_2025 = pd.date_range(start='2025-01-01 00:00', end='2025-12-31 23:00', freq='H')


Saved 2025 predictions for Vastral as sarima_vastral_2025.csv

Saving all regions metrics...
All regions tuning metrics saved as sarima_model_metrics_2024.csv
Process completed successfully.
